In [19]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

## 1. Import FashionMNIST dataset

In [20]:
# Download training data from open datasets
training_data = datasets.FashionMNIST(
    root = 'data',
    train = True,
    download = True,
    transform = ToTensor()
)

In [21]:
# Download test data 
test_data = datasets.FashionMNIST(
    root = 'data',
    train = False,
    download = True,
    transform = ToTensor()
)

### Pass datasets as arguments to DataLoader

In [22]:
BATCH_SIZE = 64

# Create data loaders
train_dataloader = DataLoader(training_data, batch_size=BATCH_SIZE)
test_dataloader = DataLoader(test_data, batch_size=BATCH_SIZE)

for X, y in test_dataloader:
    print(f"Shape of X: [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X: [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


## 2. Creating Models

In [23]:
torch.accelerator.current_accelerator().type

'cuda'

In [24]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else 'cpu'
print(f"Using {device} device")

Using cuda device


In [25]:
# Define model

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()

        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )
    
    def forward(self, X):
        X = self.flatten(X)
        logits = self.linear_relu_stack(X)
        return logits

model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


### Optimizing model parameters

In [26]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr = 1e-3)

In a single loop, the model makes predictions on the training dataset (fed to it in batches) and and backpropagates the prediction error to adjust model's parameters

In [27]:
def train(dataLoader, model, loss_fn, optimizer):
    size = len(dataLoader.dataset)
    model.train()

    for batch, (X, y) in enumerate(dataLoader):
        X, y = X.to(device), y.to(device)

        #Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        #Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"Loss: {loss:>7f} [{current:>5d} / {size:>5d}]")

We also check the model's performance the test dataset to ensure its learning

In [28]:
def test(dataLoader, model, loss_fn):
    size = len(dataLoader.dataset)
    num_batches = len(dataLoader)

    model.eval()
    test_loss, correct = 0, 0

    with torch.no_grad():
        for X, y in dataLoader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

- Training is conducted over several iterations (epochs)
- During each epoch, the model learns parameters to make better predictions.
- We print the model’s accuracy and loss at each epoch; we’d like to see the accuracy increase and the loss decrease with every epoch.



## 3. Train the Model

In [29]:
epochs = 5

for t in range(epochs):
    print(f"Epoch {t + 1}\n")
    print("="*50)
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Training complete!")

Epoch 1

Loss: 2.313768 [   64 / 60000]
Loss: 2.301614 [ 6464 / 60000]
Loss: 2.279703 [12864 / 60000]
Loss: 2.273783 [19264 / 60000]
Loss: 2.255387 [25664 / 60000]
Loss: 2.226219 [32064 / 60000]
Loss: 2.237342 [38464 / 60000]
Loss: 2.213946 [44864 / 60000]
Loss: 2.220531 [51264 / 60000]
Loss: 2.182131 [57664 / 60000]
Test Error: 
 Accuracy: 47.4%, Avg loss: 2.174194 

Epoch 2

Loss: 2.190558 [   64 / 60000]
Loss: 2.177370 [ 6464 / 60000]
Loss: 2.127329 [12864 / 60000]
Loss: 2.143309 [19264 / 60000]
Loss: 2.090193 [25664 / 60000]
Loss: 2.042790 [32064 / 60000]
Loss: 2.073234 [38464 / 60000]
Loss: 2.016729 [44864 / 60000]
Loss: 2.023527 [51264 / 60000]
Loss: 1.949883 [57664 / 60000]
Test Error: 
 Accuracy: 61.4%, Avg loss: 1.941638 

Epoch 3

Loss: 1.974702 [   64 / 60000]
Loss: 1.945074 [ 6464 / 60000]
Loss: 1.841528 [12864 / 60000]
Loss: 1.878256 [19264 / 60000]
Loss: 1.757064 [25664 / 60000]
Loss: 1.714953 [32064 / 60000]
Loss: 1.738660 [38464 / 60000]
Loss: 1.651147 [44864 / 60000]
L

## 4. Predicting

In [31]:
classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

model.eval()
X, y = test_data[0][0], test_data[0][1]

with torch.no_grad():
    X = X.to(device)
    pred = model(X)
    predicted, actual = classes[pred[0].argmax(0)], classes[y]
    print(f'Predicted: "{predicted}", Actual: "{actual}"')

Predicted: "Ankle boot", Actual: "Ankle boot"
